アノテーションのときに画面外をアノテーションしたらエラーになるのでそれを直してる

In [1]:
import os

# ▼▼▼ このルートディレクトリのパスをご自身の環境に合わせてください ▼▼▼
root_labels_dir = '/home/YOLO/0708_maesyori/datasets/labels'
# ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲

print(f"指定されたディレクトリ以下のラベルファイルをチェックします: {root_labels_dir}")

# os.walkを使って、指定したディレクトリ以下の全てのサブディレクトリを再帰的に探索
for dirpath, _, filenames in os.walk(root_labels_dir):
    print(f"\n--- フォルダをチェック中: {dirpath} ---")
    
    corrected_file_count = 0
    for filename in filenames:
        # .txtファイルのみを対象にする
        if filename.endswith(".txt"):
            filepath = os.path.join(dirpath, filename)
            
            corrected_lines = []
            needs_correction = False
            
            try:
                with open(filepath, 'r') as f:
                    lines = f.readlines()

                for line in lines:
                    parts = line.strip().split()
                    if not parts:
                        corrected_lines.append('') # 空行を保持
                        continue

                    class_id = parts[0]
                    coords = parts[1:]
                    
                    # 座標値をチェックして修正
                    current_coords_corrected = False
                    corrected_coords = []
                    for coord in coords:
                        c_float = float(coord)
                        if c_float < 0.0:
                            c_float = 0.0
                            current_coords_corrected = True
                        elif c_float > 1.0:
                            c_float = 1.0
                            current_coords_corrected = True
                        corrected_coords.append(str(c_float))
                    
                    if current_coords_corrected:
                        needs_correction = True

                    corrected_lines.append(f"{class_id} {' '.join(corrected_coords)}")

                # 修正があった場合のみファイルを上書き
                if needs_correction:
                    with open(filepath, 'w') as f:
                        f.write('\n'.join(corrected_lines))
                    corrected_file_count += 1

            except Exception as e:
                print(f"  [エラー] ファイル処理中に問題が発生しました: {filename}, エラー: {e}")

    if corrected_file_count > 0:
        print(f"-> {corrected_file_count} 個のファイルを修正しました。")
    else:
        print("-> このフォルダに修正が必要なファイルはありませんでした。")

print("\n✅ すべてのフォルダのチェックが完了しました。")

指定されたディレクトリ以下のラベルファイルをチェックします: /home/YOLO/0708_maesyori/datasets/labels

--- フォルダをチェック中: /home/YOLO/0708_maesyori/datasets/labels ---
-> このフォルダに修正が必要なファイルはありませんでした。

--- フォルダをチェック中: /home/YOLO/0708_maesyori/datasets/labels/test ---
-> 1 個のファイルを修正しました。

--- フォルダをチェック中: /home/YOLO/0708_maesyori/datasets/labels/0708_maesyori ---
-> 10 個のファイルを修正しました。

--- フォルダをチェック中: /home/YOLO/0708_maesyori/datasets/labels/val ---
-> 2 個のファイルを修正しました。

✅ すべてのフォルダのチェックが完了しました。
